In [ ]:
# 导入所需库
import sys
import os
import warnings
warnings.filterwarnings('ignore')
# windows路径库
from pathlib import Path
# 数据分析库
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
# 正则模块
import re 

In [ ]:
# # 获取程序目录文件
# existing_path = Path(os.getcwd())
# existing_path

### 初始化结果列表，采集目标文件路径

In [ ]:
# 备忘：[period,ware_type,ware_name,peihuo_efficiency,shouhuo_efficiency,dc_efficiency,zhuangxiang_amount]
# 字段名['period','ware_type','ware_name','peihuo_efficiency','shouhuo_efficiency','dc_efficiency','zhuangxiang_amount']

## 初始化结果列表
# e2取仓库信息录入ware_list
ware_list = []
# e1取物流人员基本效率信息，e2取# 取e2中年月期间、仓库种类、配货效率、收货效率、DC效率，装箱金额；从文件名中取仓库名称（未清洗），e2的结果插入e1列
# 物流人员绩效最终表——e3取配送额，efficiency_list和e3合并为efficiency_result
efficiency_result = []
# 记录无法录入文件的路径
error_path = []

In [ ]:
# 遍历所有文件，先筛选出所有xlsx文件，遍历取文件名判断是否含绩效、仓库名等；
path_collect = []
existing_path = Path(input('输入文件目标目录：'))
pattern = re.compile(r'(?=.*绩效)[\u4e00-\u9fa5]+[-仓]')

for path in (existing_path).rglob('*.xls*'):
    if re.search(pattern,path.stem) == None:
            print(f'剔除 {path}')
    else:
        path_collect.append(path)
        print(f'采集 {path}')

print("有效文件总数：",len(path_collect),end = '\n')

for path in path_collect:
    print(path,sep= '\n')

### 遍历筛选后的路径，针对每个路径数据处理并拼合

In [ ]:
# 开启循环
for path in path_collect:
    try:
        # 读取文件
        e1 = pd.read_excel(path,skiprows = 8,engine = 'openpyxl')
        
        # 改名
        e1 = e1.rename(columns = {'工号\n（务必准确）':'工号'})
        e2 = pd.read_excel(path,nrows = 5,usecols=[1,2,3,4,5,6],engine = 'openpyxl',header = None)
        
        # 取e2中年月期间、仓库种类、配货效率、收货效率、DC效率，装箱金额；从文件名中取仓库名称（未清洗）
        
        period = e2.iloc[0,0]
        ware_type = e2.iloc[0,3]
        pattern = re.compile(r'(?=.*绩效)[\u4e00-\u9fa50-9A-Z]+仓')
        ware_name_uncleaned = re.search(pattern,str(path.stem)).group()
        ware_name = ware_name_uncleaned.replace('-','')
        # 取仓库指标
        peihuo_efficiency = e2.iloc[1,3]
        shouhuo_efficiency = e2.iloc[2,3]
        peihuo_entire_efficiency = e2.iloc[1,5]
        shouhuo_entire_efficiency = e2.iloc[2,5]
        dc_efficiency = e2.iloc[3,1]
        zhuangxiang_amount = e2.iloc[1,1]
        
        # 生成结果
        ware = [period,ware_type,ware_name,peihuo_efficiency,shouhuo_efficiency,peihuo_entire_efficiency,shouhuo_entire_efficiency,dc_efficiency,zhuangxiang_amount]
        
        # 合并结果——通过e1添加新列（结果取e2）
        e1.insert(loc = 0,column = '是否国际',value = ware_type)
        e1.insert(loc = 0,column = '仓库名称',value = ware_name)
        e1.insert(loc = 0,column = '年月期间',value = period)
        
        # 读取配货金额，"效率数据sheet"
        e3 = pd.read_excel(path,sheet_name = '效率数据',engine = 'openpyxl',usecols = ['工号','配货金额（国内仓库）','配货金额（原箱）','配货金额（拆零）'],skiprows = 1)
        
        # 国际仓库时读取协助岗-绩效核算明细sheet，取
        if ware_type == '是':
            e4 = pd.read_excel(path,sheet_name = '协助岗-绩效核算明细',engine = 'openpyxl',usecols = ['工号\n（务必准确）','系统导出效率数据\n（原箱）','系统导出效率数据（协助原箱）','效率\n（原箱）'],skiprows = 8).dropna(subset=['工号\n（务必准确）'])
            if len(e4) >= 1:
                e3 = pd.merge(e3,e4,left_on = '工号',right_on = '工号\n（务必准确）',how = 'left')

        # 将配货金额匹配e1效率数据，生成最终结果表
        # e1['工号'] = e1['工号'].astype(str)
        # e3['工号'] = e3['工号'].astype(str)
        e = pd.merge(e1,e3,left_on = '工号',right_on = '工号',how = 'left')
        
        # 添加本次处理结果
        ware_list.append(ware)
        efficiency_result.append(e)
        print(f'{path} processed')
        
    except Exception as e:
        error_path.append(str(path))
        print(f'error occur! {str(path)}',end='\n')

### 结果生成

In [ ]:
# 生成结果
# existing_path = Path(r'D:\Users\Desktop\物流更新\2025.6物流中心业绩表18仓')
efficiency_list = pd.concat(efficiency_result).dropna(subset = ['工号'])
ware_list = pd.DataFrame(np.array(ware_list).reshape(-1,9),columns = ['period','ware_type','ware_name','peihuo_efficiency','shouhuo_efficiency','peihuo_entire_efficiency','shouhuo_entire_efficiency','dc_efficiency','zhuangxiang_amount'])
ware_list = ware_list.drop(columns = ['shouhuo_entire_efficiency'])

# 历史名称统一
efficiency_list['花名册仓库'] = efficiency_list['仓库名称']
efficiency_list['花名册仓库'][efficiency_list['花名册仓库']=='肇庆潮玩仓'] = '肇庆TOPTOY仓'
efficiency_list['花名册仓库'][efficiency_list['花名册仓库']=='沈阳潮玩仓'] = '沈阳TOPTOY仓'
efficiency_list['统一仓库名称'] = efficiency_list['花名册仓库']
# 次序重排
efficiency_list_result = efficiency_list[['年月期间','仓库名称','是否国际','花名册仓库','统一仓库名称','工号','姓名','岗位名称\n（实际工作岗位）','标准工时','考勤总工时','本职岗位/拆零工时','协助同岗位工时','培训工时','绩效评分\n(固定绩效基数）','负激励金额','系统导出效率数据\n（国内仓库）','效率\n（国内仓库）','系统导出效率数据\n（拆零）','效率\n（拆零）','效率奖金基数\n（国内仓库/拆零）','不足工时配货效率金额','不足工时协助奖金（协助同岗位）','不足工时协助奖金（协助其他岗位/原箱）','不足工时培训奖金','准确\n率奖基数','不足工时准确率奖金','正负绝对值奖金','DC效率奖金','配货效率奖金','装箱金额','提成奖金合计','协助绩效奖金金额','固定绩效奖金基数\n（评分制，如无请更新绩效标准表）','绩效奖金金额','绩效奖金合计','年度大盘奖金','奖金发放总额','实际出勤小时\n（提交集团）','配货金额（国内仓库）','配货金额（原箱）','配货金额（拆零）','系统导出效率数据\n（原箱）','系统导出效率数据（协助原箱）','效率\n（原箱）']]
# 更名
efficiency_list_result.rename(columns={'是否国际':'是否国际仓库','岗位名称\n（实际工作岗位）':'岗位名称（实际工作岗位）','绩效评分\n(固定绩效基数）':'绩效评分（固定绩效基数）','系统导出效率数据\n（国内仓库）':'系统导出效率数据（国内仓库）','效率\n（国内仓库）':'效率（国内仓库）','系统导出效率数据\n（拆零）':'系统导出效率数据（拆零）','效率\n（拆零）':'效率（拆零）','效率奖金基数\n（国内仓库/拆零）':'效率奖金基数（国内仓库/拆零）','准确\n率奖基数':'准确率奖基数','固定绩效奖金基数\n（评分制，如无请更新绩效标准表）':'固定绩效奖金基数（评分制，如无请更新绩效标准表）','实际出勤小时\n（提交集团）':'实际出勤小时（提交集团）','系统导出效率数据\n（原箱）':'系统导出效率数据（原箱）','效率\n（原箱）':'效率（原箱）'},inplace= True)

# 添加五个字段，值均填充为空值
efficiency_list_result[['总人力成本','入职日期','入职第几个月','是否潜力新员工','绩效奖金']]=0
efficiency_list_result.fillna(0,inplace=True)

# 导出到程序所在目录文件夹
efficiency_list_result.to_excel(existing_path / 'efficiency_list.xlsx',index = False)
ware_list.to_excel(existing_path / 'ware_list.xlsx',index = False)

# 打印无法录入的文件名单
for i in error_path:
    print(i)

### 对结果分表复核（给各仓计统岗）

In [ ]:
# 汇总后再分表，可用于供业务核对汇报结果是否完整
# 定义excel处理后路径，后读取分表
final_result_path = Path(r'D:\Users\Desktop\物流数据汇总\物流人效明细.xlsx')
final_result_path2 = Path(r'D:\Users\Desktop\物流数据汇总\物流仓库明细.xlsx')
save_path = Path(r'D:\Users\Desktop\物流数据汇总\人效拆分数据供核对')
save_path2 = Path(r'D:\Users\Desktop\物流数据汇总\仓库拆分数据供核对')
final_result = pd.read_excel(final_result_path,header = 0)
final_result2 = pd.read_excel(final_result_path2,header=0,usecols = list(range(0,10)))

In [ ]:
# 人效明细分表
grouped = final_result.groupby('统一仓库名称')
for name,group in grouped:
    group.to_excel(save_path / f'{name}_人效明细.xlsx',index = False)

In [ ]:
# 仓库月度数据分表
grouped2 = final_result2.groupby('统一仓库名称')
for name,group in grouped2:
    group.to_excel(save_path2 / f'{name}_仓库明细.xlsx',index = False)